In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("ExpenseTransactionAnalysis") \
    .getOrCreate()

In [2]:
df = spark.read.csv(
    "/content/expense_data_cleaned.csv",
    header=True,
    inferSchema=True
)
print("Total transactions loaded:", df.count())
df.printSchema()

Total transactions loaded: 94
root
 |-- date: date (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- user_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- month: timestamp (nullable = true)



In [3]:
monthly_user_spend = df.groupBy("month", "user_name") \
    .agg(F.round(F.sum("amount"), 2).alias("total_spend"),
         F.count("*").alias("transaction_count")) \
    .orderBy("month", "user_name")

print("\nMonthly spend by user:")
monthly_user_spend.show(20, truncate=False)


Monthly spend by user:
+-------------------+------------+-----------+-----------------+
|month              |user_name   |total_spend|transaction_count|
+-------------------+------------+-----------+-----------------+
|2026-05-01 00:00:00|Arjun Mehta |13619.5    |10               |
|2026-05-01 00:00:00|Karthik Iyer|9537.32    |8                |
|2026-05-01 00:00:00|Priya Nair  |10086.24   |10               |
|2026-05-01 00:00:00|Rohan Verma |14437.11   |10               |
|2026-05-01 00:00:00|Sneha Reddy |10138.58   |11               |
|2026-06-01 00:00:00|Arjun Mehta |8846.66    |8                |
|2026-06-01 00:00:00|Karthik Iyer|17540.08   |10               |
|2026-06-01 00:00:00|Priya Nair  |6975.36    |8                |
|2026-06-01 00:00:00|Rohan Verma |12293.07   |11               |
|2026-06-01 00:00:00|Sneha Reddy |6664.18    |8                |
+-------------------+------------+-----------+-----------------+



In [4]:
user_stats = df.groupBy("user_name") \
    .agg(F.round(F.avg("amount"), 2).alias("avg_amount"),
         F.round(F.stddev("amount"), 2).alias("stddev_amount"))

print("\nPer-user transaction stats:")
user_stats.show(truncate=False)


Per-user transaction stats:
+------------+----------+-------------+
|user_name   |avg_amount|stddev_amount|
+------------+----------+-------------+
|Rohan Verma |1272.87   |906.18       |
|Arjun Mehta |1248.12   |702.61       |
|Karthik Iyer|1504.3    |1011.87      |
|Sneha Reddy |884.36    |527.05       |
|Priya Nair  |947.87    |651.86       |
+------------+----------+-------------+



In [5]:
df_with_stats = df.join(user_stats, on="user_name") \
    .withColumn("anomaly_threshold", F.round(F.col("avg_amount") + 2 * F.col("stddev_amount"), 2)) \
    .withColumn("is_anomaly", F.col("amount") > F.col("anomaly_threshold"))

anomalies = df_with_stats.filter(F.col("is_anomaly")) \
    .select("date", "user_name", "category", "amount", "avg_amount", "anomaly_threshold") \
    .orderBy(F.col("amount").desc())

print("\nFlagged unusual transactions:")
anomalies.show(truncate=False)


Flagged unusual transactions:
+----------+------------+---------+-------+----------+-----------------+
|date      |user_name   |category |amount |avg_amount|anomaly_threshold|
+----------+------------+---------+-------+----------+-----------------+
|2026-06-11|Karthik Iyer|Shopping |4273.19|1504.3    |3528.04          |
|2026-05-22|Rohan Verma |Shopping |3917.24|1272.87   |3085.23          |
|2026-05-18|Priya Nair  |Shopping |2732.17|947.87    |2251.59          |
|2026-06-10|Sneha Reddy |Groceries|2052.72|884.36    |1938.46          |
+----------+------------+---------+-------+----------+-----------------+



In [6]:
user_anomaly_summary = anomalies.groupBy("user_name") \
    .agg(F.count("*").alias("unusual_transaction_count"),
         F.round(F.max("amount"), 2).alias("largest_unusual_amount")) \
    .orderBy(F.col("unusual_transaction_count").desc())

print("\nUsers with potential unusual spending:")
user_anomaly_summary.show(truncate=False)


Users with potential unusual spending:
+------------+-------------------------+----------------------+
|user_name   |unusual_transaction_count|largest_unusual_amount|
+------------+-------------------------+----------------------+
|Rohan Verma |1                        |3917.24               |
|Karthik Iyer|1                        |4273.19               |
|Sneha Reddy |1                        |2052.72               |
|Priya Nair  |1                        |2732.17               |
+------------+-------------------------+----------------------+



In [7]:
monthly_user_spend.toPandas().to_csv("/content/monthly_user_spend.csv", index=False)
anomalies.toPandas().to_csv("/content/expense_anomalies.csv", index=False)
user_anomaly_summary.toPandas().to_csv("/content/users_with_unusual_spending.csv", index=False)


In [8]:
spark.stop()